<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/uni_mol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit
!pip install unimol_tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.9/120.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 10.3 MB/s eta 0:00:00


In [ ]:
import os
import torch
import numpy as np
from rdkit import Chem
from unimol_tools import UniMolRepr
import warnings
from sklearn.metrics import roc_auc_score, matthews_corrcoef, average_precision_score, f1_score
warnings.filterwarnings('ignore')

SDF_BASE_DIR = "/content/CV_Folds"
OUTPUT_DIR = "./data/embeddings"
FOLDS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Loading Uni-Mol...")
unimol = UniMolRepr(data_type='molecule', remove_hs=False)

def process_fold_files(active_sdf_path, negative_sdf_path, output_name):
    """Reads both active and negative SDFs, combines them, and generates Uni-Mol embeddings."""

    # Initialize as a dictionary of lists
    custom_data = {
        "atoms": [],
        "coordinates": []
    }
    labels = []

    # Helper function to read an SDF and assign a fixed label based on the file source
    def read_sdf(sdf_path, assigned_label):
        if not os.path.exists(sdf_path):
            print(f"  -> Warning: File not found: {sdf_path}")
            return

        supplier = Chem.SDMolSupplier(sdf_path, removeHs=False, sanitize=True)
        for mol in supplier:
            if mol is None:
                continue

            labels.append(assigned_label)
            conf = mol.GetConformer()
            atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
            coords = conf.GetPositions().tolist()

            # Append to the respective lists in the dictionary
            custom_data["atoms"].append(atoms)
            custom_data["coordinates"].append(coords)

    # Read actives (Label 1.0) and negatives (Label 0.0)
    read_sdf(active_sdf_path, 1.0)
    read_sdf(negative_sdf_path, 0.0)

    if len(labels) == 0:
        print(f"No valid molecules found for {output_name}. Skipping.")
        return

    print(f"Extracting representations for {output_name} ({len(labels)} compounds)...")

  # Get representations
    repr_output = unimol.get_repr(custom_data, return_atomic_reprs=False)


    # This check extracts the [CLS] embeddings regardless of the structure.
    if isinstance(repr_output, dict):
        # Case 1: It returned a dictionary
        cls_embeddings = np.array(repr_output['cls_repr'])

    elif isinstance(repr_output, list) or isinstance(repr_output, np.ndarray):
        if len(repr_output) > 0 and isinstance(repr_output[0], dict):
            # Case 2: It returned a list of dictionaries
            cls_embeddings = np.array([item['cls_repr'] for item in repr_output])
        else:
            # Case 3: It returned the list of embeddings directly
            cls_embeddings = np.array(repr_output)

    else:
        raise ValueError(f"Unexpected output type from Uni-Mol: {type(repr_output)}")

    # Check shape to ensure it matches (Num_Molecules, Hidden_Dimension)
    print(f"Extracted embeddings shape: {cls_embeddings.shape}")

    # Convert to PyTorch tensors
    tensor_embeddings = torch.tensor(cls_embeddings, dtype=torch.float32)
    tensor_labels = torch.tensor(labels, dtype=torch.float32)

    # Save the combined tensor file
    out_file = os.path.join(OUTPUT_DIR, f"{output_name}.pt")
    torch.save({"embeddings": tensor_embeddings, "labels": tensor_labels}, out_file)
    print(f"Saved {tensor_embeddings.shape[0]} compounds to {out_file}\n")

if __name__ == "__main__":
    for i in range(1, FOLDS + 1):
        print(f"Processing Fold {i}:")

        # Define paths based on your specific naming convention
        train_actives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Train_Actives_3D.sdf")
        train_negatives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Train_Negatives_3D.sdf")

        val_actives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Val_Actives_3D.sdf")
        val_negatives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Val_Negatives_3D.sdf")

        # Process and combine Train set
        process_fold_files(train_actives, train_negatives, f"Fold_{i}_Train")

        # Process and combine Val set
        process_fold_files(val_actives, val_negatives, f"Fold_{i}_Val")

print("All Uni-Mol embeddings generated successfully!")

2026-07-12 20:39:14 | unimol_tools/weights/weighthub.py | 54 | INFO | Uni-Mol Tools | Weights will be downloaded to default directory: /usr/local/lib/python3.12/dist-packages/unimol_tools/weights
INFO:Uni-Mol Tools:Weights will be downloaded to default directory: /usr/local/lib/python3.12/dist-packages/unimol_tools/weights
2026-07-12 20:39:14 | unimol_tools/weights/weighthub.py | 71 | INFO | Uni-Mol Tools | Downloading mol_pre_all_h_220816.pt
INFO:Uni-Mol Tools:Downloading mol_pre_all_h_220816.pt


Loading Uni-Mol...


DEBUG:httpcore.connection:connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=3 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c2c9430f710>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x7c2c93f827d0> server_hostname='huggingface.co' timeout=3
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c2c94c43170>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'GET']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'GET']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'GET']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'application/json; charset=utf-8'), (b'T

DEBUG:filelock:Attempting to acquire lock 136530905686400 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Lock 136530905686400 acquired on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Attempting to release lock 136530905686400 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Lock 136530905686400 released on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Attempting to acquire lock 136530905686400 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/download/mol_pre_all_h_220816.pt.lock
DEBUG:filelock:Lock 136530905686400 acquired on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/download/mol_pre_all_h_220816.pt.lock


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Extracting representations for Fold_5_Train (550 compounds)...


100%|██████████| 18/18 [01:04<00:00,  3.58s/it]
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[PAD]', index is 2.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[CLS]', index is 0.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[SEP]', index is 3.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[UNK]', index is 1.
2026-07-12 20:45:59 | unimol_tools/tasks/trainer.py | 103 | INFO | Uni-Mol Tools | Using CPU.
INFO:Uni-Mol Tools:Using CPU.


Extracted embeddings shape: (550, 512)
Saved 550 compounds to ./data/embeddings/Fold_5_Train.pt

Extracting representations for Fold_5_Val (135 compounds)...


100%|██████████| 5/5 [00:14<00:00,  2.96s/it]

Extracted embeddings shape: (135, 512)
Saved 135 compounds to ./data/embeddings/Fold_5_Val.pt

All Uni-Mol embeddings generated successfully!


In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=0.40, gamma=2.0, smoothing=0.1):
        super(FocalLossWithSmoothing, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        # 1. Apply Label Smoothing
        targets_smoothed = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing

        # 2. Compute base BCE Loss
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets_smoothed, reduction='none')

        # 3. Compute pt (probability of the correct class)
        pt = torch.exp(-BCE_loss)

        # 4. Construct the alpha_t term based on the original (unsmoothed) targets

        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)

        # 5. Calculate Focal Loss
        focal_weight = alpha_t * (1 - pt) ** self.gamma

        return torch.mean(focal_weight * BCE_loss)

class UniMolMLPHead(nn.Module):
    def __init__(self, input_dim=512, hidden_dim_1=256, hidden_dim_2=64, dropout_rate=0.5):
        super(UniMolMLPHead, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.BatchNorm1d(hidden_dim_1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.BatchNorm1d(hidden_dim_2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim_2, 1)
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x).squeeze(-1)

In [37]:
import os
import torch
from torch.utils.data import Dataset

class UniMolFoldDataset(Dataset):
    def __init__(self, fold_idx, is_train, base_dir="./data/embeddings"):
        phase = "Train" if is_train else "Val"
        file_path = os.path.join(base_dir, f"Fold_{fold_idx}_{phase}.pt")
        data = torch.load(file_path)
        self.embeddings = data["embeddings"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


In [38]:
def train_fold(fold_idx, epochs=150, batch_size=32, lr=5e-4):
    print(f"Starting Fold {fold_idx}/5")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    train_dataset = UniMolFoldDataset(fold_idx, is_train=True)
    val_dataset = UniMolFoldDataset(fold_idx, is_train=False)


    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


    model = UniMolMLPHead(input_dim=512).to(device)
    criterion = FocalLossWithSmoothing(alpha=0.40, gamma=2.0, smoothing=0.1)


    # Aggressive Weight Decay (1e-2) and patience (20)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=20)


    # Track Best AUC
    best_auc = 0.0
    best_metrics = {}


    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for embeddings, labels in train_loader:
            if embeddings.size(0) == 1: continue
            embeddings, labels = embeddings.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(embeddings)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * embeddings.size(0)


        train_loss /= len(train_loader.dataset)


        model.eval()
        val_loss = 0.0
        all_logits = []
        all_labels = []


        with torch.no_grad():
            for embeddings, labels in val_loader:
                embeddings, labels = embeddings.to(device), labels.to(device)
                logits = model(embeddings)
                loss = criterion(logits, labels)
                val_loss += loss.item() * embeddings.size(0)
                all_logits.extend(logits.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())


        val_loss /= len(val_loader.dataset)
        raw_probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
        raw_labels = np.array(all_labels)


        # Test-Time Conformer Consensus (Boltzmann Averaging)
        NUM_CONFORMERS = 5


        if len(raw_probs) % NUM_CONFORMERS == 0:
            probs = raw_probs.reshape(-1, NUM_CONFORMERS).mean(axis=1)
            all_labels = raw_labels.reshape(-1, NUM_CONFORMERS)[:, 0]
        else:
            print("\n[WARNING] Validation set not divisible by 5. Test-Time Consensus skipped!\n")
            probs = raw_probs
            all_labels = raw_labels


        # Calculate Threshold-Independent Metrics
        try:
            auc = roc_auc_score(all_labels, probs)
            pr_auc = average_precision_score(all_labels, probs)
        except ValueError:
            auc, pr_auc = 0.0, 0.0


        # Dynamic Optimal Thresholding for MCC & F1
        best_fold_mcc = -1.0
        best_fold_f1 = 0.0
        best_thresh = 0.5


        for thresh in np.arange(0.20, 0.85, 0.05):
            temp_preds = (probs > thresh).astype(int)
            try:
                temp_mcc = matthews_corrcoef(all_labels, temp_preds)
                temp_f1 = f1_score(all_labels, temp_preds)
            except ValueError:
                temp_mcc, temp_f1 = 0.0, 0.0


            if temp_mcc > best_fold_mcc:
                best_fold_mcc = temp_mcc
                best_fold_f1 = temp_f1
                best_thresh = thresh


        scheduler.step(auc)


        if auc > best_auc:
            best_auc = auc
            best_metrics = {"auc": auc, "pr_auc": pr_auc, "f1": best_fold_f1, "mcc": best_fold_mcc, "thresh": best_thresh}
            torch.save(model.state_dict(), f"best_model_fold_{fold_idx}.pth")


        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d} | Loss: {val_loss:.4f} | AUC: {auc:.4f} | PR-AUC: {pr_auc:.4f} | F1: {best_fold_f1:.4f} | MCC: {best_fold_mcc:.4f}")


    print(f"Fold {fold_idx} Final -> AUC: {best_metrics['auc']:.4f} | PR-AUC: {best_metrics['pr_auc']:.4f} | F1: {best_metrics['f1']:.4f} | MCC: {best_metrics['mcc']:.4f}")
    return best_metrics


In [39]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import torch.optim as optim

if __name__ == "__main__":
    metrics_history = []

    for i in range(1, 6):
        fold_metrics = train_fold(fold_idx=i, epochs=150, batch_size=32, lr=5e-4)
        metrics_history.append(fold_metrics)

    print("3D Uni-Mol 2 Base Cross-Validation Results:")


    for m_key in ["auc", "pr_auc", "f1", "mcc"]:
        values = [m[m_key] for m in metrics_history]
        name = m_key.upper().replace("_", " ")
        print(f"Average {name:7} : {np.mean(values):.4f} +/- {np.std(values):.4f}")

Starting Fold 1/5
Epoch 001 | Loss: 0.0425 | AUC: 0.8957 | PR-AUC: 0.9708 | F1: 0.9787 | MCC: 0.8756
Epoch 020 | Loss: 0.0510 | AUC: 0.9826 | PR-AUC: 0.9963 | F1: 0.9545 | MCC: 0.8076
Epoch 040 | Loss: 0.0621 | AUC: 0.9391 | PR-AUC: 0.9878 | F1: 0.9302 | MCC: 0.7372
Epoch 060 | Loss: 0.0664 | AUC: 0.9391 | PR-AUC: 0.9878 | F1: 0.9302 | MCC: 0.7372
Epoch 080 | Loss: 0.0667 | AUC: 0.9391 | PR-AUC: 0.9878 | F1: 0.9302 | MCC: 0.7372
Epoch 100 | Loss: 0.0708 | AUC: 0.9391 | PR-AUC: 0.9878 | F1: 0.9302 | MCC: 0.7372
Epoch 120 | Loss: 0.0690 | AUC: 0.9391 | PR-AUC: 0.9878 | F1: 0.9302 | MCC: 0.7372
Epoch 140 | Loss: 0.0712 | AUC: 0.9391 | PR-AUC: 0.9878 | F1: 0.9302 | MCC: 0.7372
Fold 1 Final -> AUC: 0.9826 | PR-AUC: 0.9963 | F1: 0.9787 | MCC: 0.8756
Starting Fold 2/5
Epoch 001 | Loss: 0.0653 | AUC: 0.9123 | PR-AUC: 0.9517 | F1: 0.9500 | MCC: 0.8389
Epoch 020 | Loss: 0.0345 | AUC: 0.9825 | PR-AUC: 0.9928 | F1: 0.9730 | MCC: 0.9234
Epoch 040 | Loss: 0.0346 | AUC: 0.9766 | PR-AUC: 0.9908 | F1: 

In [44]:
import os
import torch
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, matthews_corrcoef, average_precision_score, f1_score
import warnings
warnings.filterwarnings('ignore')


def train_y_randomization_fold(fold_idx, scrambled_train_labels, epochs=50, batch_size=32, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = UniMolFoldDataset(fold_idx, is_train=True)
    train_dataset.labels = scrambled_train_labels
    val_dataset = UniMolFoldDataset(fold_idx, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = UniMolMLPHead(input_dim=512).to(device)
    criterion = FocalLossWithSmoothing(alpha=0.75, gamma=2.0, smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    NUM_CONFORMERS = 50

    for epoch in range(epochs):
        model.train()
        for embeddings, labels in train_loader:
            if embeddings.size(0) == 1: continue
            embeddings, labels = embeddings.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(embeddings)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

    # -Evaluate at the end
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for embeddings, labels in val_loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            logits = model(embeddings)
            all_logits.extend(logits.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    raw_probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
    raw_labels = np.array(all_labels)

    # Test-Time Conformer Consensus
    if len(raw_probs) % NUM_CONFORMERS == 0:
        probs = raw_probs.reshape(-1, NUM_CONFORMERS).mean(axis=1)
        final_labels = raw_labels.reshape(-1, NUM_CONFORMERS)[:, 0]
    else:
        probs = raw_probs
        final_labels = raw_labels

    preds = (probs > 0.5).astype(int)

    try:
        auc = roc_auc_score(final_labels, probs)
        pr_auc = average_precision_score(final_labels, probs)
        mcc = matthews_corrcoef(final_labels, preds)
        f1 = f1_score(final_labels, preds)
    except ValueError:
        auc, pr_auc, mcc, f1 = 0.5, 0.0, 0.0, 0.0

    # Return the exact state at the end of training
    return {"auc": auc, "mcc": mcc, "pr_auc": pr_auc, "f1": f1}
if __name__ == "__main__":
    N_ITERATIONS = 50
    print(f"Starting 3D Y-Randomization Validation ({N_ITERATIONS} Permutations)...")

    all_rand_aucs = []
    all_rand_mccs = []
    all_rand_prs = []
    all_rand_f1s = []

    for iteration in range(N_ITERATIONS):
        iter_aucs = []
        iter_mccs = []
        iter_prs = []
        iter_f1s = []

        # Run across all 5 folds for this single permutation iteration
        for i in range(1, 6):
            # 1. Load the original train dataset just to get its structure and labels
            temp_dataset = UniMolFoldDataset(i, is_train=True)
            original_labels = temp_dataset.labels

            # 2. Extract molecule-level labels (Assuming exactly 5 conformers per molecule)
            num_mols = len(original_labels) // 5
            mol_labels = original_labels.view(num_mols, 5)[:, 0]

            # 3. Shuffle labels at the molecule level
            perm = torch.randperm(num_mols)
            shuffled_mol_labels = mol_labels[perm]

            # 4. Broadcast the shuffled labels back to the 5 conformers
            scrambled_conformer_labels = shuffled_mol_labels.unsqueeze(1).repeat(1, 5).view(-1)

            # 5. Train the fold with these explicitly scrambled labels
            res = train_y_randomization_fold(fold_idx=i, scrambled_train_labels=scrambled_conformer_labels, epochs=50)
            iter_aucs.append(res['auc'])
            iter_mccs.append(res['mcc'])
            iter_prs.append(res['pr_auc'])
            iter_f1s.append(res['f1'])

        # Calculate means for this permutation iteration
        mean_iter_auc = np.mean(iter_aucs)
        mean_iter_mcc = np.mean(iter_mccs)
        mean_iter_pr = np.mean(iter_prs)
        mean_iter_f1 = np.mean(iter_f1s)

        all_rand_aucs.append(mean_iter_auc)
        all_rand_mccs.append(mean_iter_mcc)
        all_rand_prs.append(mean_iter_pr)
        all_rand_f1s.append(mean_iter_f1)

        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(f"Run {iteration+1}/{N_ITERATIONS} Mean Metrics | AUC: {mean_iter_auc:.4f} | PR-AUC: {mean_iter_pr:.4f} | F1: {mean_iter_f1:.4f} | MCC: {mean_iter_mcc:.4f}")

    print("\nY-Randomization Final Results: ")
    print(f"Average Random AUC : {np.mean(all_rand_aucs):.4f} +/- {np.std(all_rand_aucs):.4f}")
    print(f"Average Random PR-AUC: {np.mean(all_rand_prs):.4f} +/- {np.std(all_rand_prs):.4f}")
    print(f"Average Random F1  : {np.mean(all_rand_f1s):.4f} +/- {np.std(all_rand_f1s):.4f}")
    print(f"Average Random MCC : {np.mean(all_rand_mccs):.4f} +/- {np.std(all_rand_mccs):.4f}")

Starting 3D Y-Randomization Validation (50 Permutations)...
Run 1/50 Mean Metrics | AUC: 0.5645 | PR-AUC: 0.6457 | F1: 0.6524 | MCC: 0.0771
Run 10/50 Mean Metrics | AUC: 0.4726 | PR-AUC: 0.6088 | F1: 0.6183 | MCC: -0.0328
Run 20/50 Mean Metrics | AUC: 0.4585 | PR-AUC: 0.6008 | F1: 0.6281 | MCC: -0.0286
Run 30/50 Mean Metrics | AUC: 0.3966 | PR-AUC: 0.5681 | F1: 0.5685 | MCC: -0.1937
Run 40/50 Mean Metrics | AUC: 0.5717 | PR-AUC: 0.6501 | F1: 0.6748 | MCC: 0.1180
Run 50/50 Mean Metrics | AUC: 0.3755 | PR-AUC: 0.5320 | F1: 0.5703 | MCC: -0.1316

Y-Randomization Final Results: 
Average Random AUC : 0.4919 +/- 0.0597
Average Random PR-AUC: 0.6110 +/- 0.0373
Average Random F1  : 0.6340 +/- 0.0332
Average Random MCC : -0.0019 +/- 0.0862


In [ ]:
import os
import torch
import numpy as np
from rdkit import Chem
from unimol_tools import UniMolRepr

# Assuming the UniMolMLPHead class is imported from your training script
from 2_train_mlp import UniMolMLPHead

BLIND_SET_SDF = "./data/Blind_Set_3D.sdf"
FOLDS = 5

def run_ensemble_inference(sdf_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Extract Embeddings for Blind Set
    print("Loading Uni-Mol and extracting blind set representations...")
    unimol = UniMolRepr(data_type='molecule', remove_hs=False)
    supplier = Chem.SDMolSupplier(sdf_path, removeHs=False, sanitize=True)

    custom_data = []
    smiles_list = []

    for mol in supplier:
        if mol is None: continue
        smiles_list.append(Chem.MolToSmiles(mol))
        conf = mol.GetConformer()
        atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
        coords = conf.GetPositions().tolist()
        custom_data.append({"atoms": atoms, "coordinates": coords})

    repr_output = unimol.get_repr(custom_data, return_atomic_reprs=False)
    blind_embeddings = torch.tensor(np.array(repr_output['cls_repr']), dtype=torch.float32).to(device)

    # 2. Load the 5 Trained Models
    models = []
    for i in range(1, FOLDS + 1):
        model = UniMolMLPHead(input_dim=512).to(device)
        model.load_state_dict(torch.load(f"best_model_fold_{i}.pth", map_location=device))
        model.eval()
        models.append(model)

    # 3. Ensemble Prediction (Consensus Scoring)
    print("Running Ensemble Inference...")
    all_probs = []

    with torch.no_grad():
        for model in models:
            logits = model(blind_embeddings)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)

    # Calculate the mean probability across all 5 models
    ensemble_probs = np.mean(all_probs, axis=0)

    # 4. Output Results
    print("\n--- ENSEMBLE BLIND SET PREDICTIONS ---")
    for smi, prob in zip(smiles_list, ensemble_probs):
        prediction = "ACTIVE" if prob >= 0.5 else "INACTIVE"
        print(f"SMILES: {smi} | Prob: {prob:.4f} | Prediction: {prediction}")

if __name__ == "__main__":
    if os.path.exists(BLIND_SET_SDF):
        run_ensemble_inference(BLIND_SET_SDF)
    else:
        print(f"Blind set SDF not found at {BLIND_SET_SDF}")